# Learning Vector Quantization (LVQ)
## TC2035 - Diseño de Redes Neuronales y Aprendizaje Profundo
### *A00838589 - Steffany Mishell Lara Muy*

Esta actividad tiene como objetivo principal aplicar el conocimiento de redes neuronales en el contexto de aprendizaje competitivo, equivalente al supervised learning en la Inteligencia Artifical normal.

Se tomará como base el tutorial brindado por el maestro con modificaciones en parámetros de sintonización, medidas de distancia el pase múltiples.



## Código

In [2]:
# How To Implement Learning Vector Quantization (LVQ) From Scratch With Python
# https://machinelearningmastery.com/implement-learning-vector-quantization-scratch-python/
#
# VERSIÓN MEJORADA CON:
# - Múltiples medidas de distancia (Euclidiana, Manhattan, Minkowski)
# - Parámetros de sintonización ajustables
# - Entrenamiento con múltiples pasadas
# - Dataset Wine de UCI

from random import seed
from random import randrange
from csv import reader
from math import sqrt, pow
import urllib.request
import os

In [3]:
# Load a CSV file
def load_csv(filename):
	dataset = list()
	with open(filename, 'r') as file:
		csv_reader = reader(file)
		for row in csv_reader:
			if not row:
				continue
			dataset.append(row)
	return dataset

In [4]:
# Download Wine dataset from UCI if not exists
def download_wine_dataset():
	filename = 'wine.csv'
	if not os.path.exists(filename):
		print("Descargando dataset Wine de UCI...")
		url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data'
		urllib.request.urlretrieve(url, filename)
		print("Dataset descargado exitosamente!")
	return filename

# Convert string column to float
def str_column_to_float(dataset, column):
	for row in dataset:
		row[column] = float(row[column].strip())

# Convert string column to integer
def str_column_to_int(dataset, column):
	class_values = [row[column] for row in dataset]
	unique = set(class_values)
	lookup = dict()
	for i, value in enumerate(unique):
		lookup[value] = i
	for row in dataset:
		row[column] = lookup[row[column]]
	return lookup

# Split a dataset into k folds
def cross_validation_split(dataset, n_folds):
	dataset_split = list()
	dataset_copy = list(dataset)
	fold_size = int(len(dataset) / n_folds)
	for i in range(n_folds):
		fold = list()
		while len(fold) < fold_size:
			index = randrange(len(dataset_copy))
			fold.append(dataset_copy.pop(index))
		dataset_split.append(fold)
	return dataset_split

In [5]:

# ============== NUEVAS MEDIDAS DE DISTANCIA ==============

# calculate the Euclidean distance between two vectors
def euclidean_distance(row1, row2):
	distance = 0.0
	for i in range(len(row1)-1):
		distance += (row1[i] - row2[i])**2
	return sqrt(distance)

# calculate the Manhattan distance between two vectors
def manhattan_distance(row1, row2):
	distance = 0.0
	for i in range(len(row1)-1):
		distance += abs(row1[i] - row2[i])
	return distance

# calculate the Minkowski distance between two vectors
def minkowski_distance(row1, row2, p=3):
	distance = 0.0
	for i in range(len(row1)-1):
		distance += pow(abs(row1[i] - row2[i]), p)
	return pow(distance, 1.0/p)

# Función para seleccionar la medida de distancia
def calculate_distance(row1, row2, distance_type='euclidean', p=3):
	if distance_type == 'euclidean':
		return euclidean_distance(row1, row2)
	elif distance_type == 'manhattan':
		return manhattan_distance(row1, row2)
	elif distance_type == 'minkowski':
		return minkowski_distance(row1, row2, p)
	else:
		return euclidean_distance(row1, row2)  # default


In [6]:

# ============== EXPERIMENTOS PRINCIPALES ==============

# Test LVQ on Wine dataset
seed(1)

# Descargar y cargar dataset Wine
filename = download_wine_dataset()
dataset = load_csv(filename)

# El dataset Wine tiene la clase en la primera columna, moverla al final
for row in dataset:
	class_value = row.pop(0)  # Quitar la primera columna (clase)
	row.append(class_value)    # Agregarla al final

# Convertir todas las columnas de features a float
for i in range(len(dataset[0])-1):
	str_column_to_float(dataset, i)

# Convertir clase a entero
str_column_to_int(dataset, len(dataset[0])-1)

print("\n" + "="*60)
print("EXPERIMENTOS CON LEARNING VECTOR QUANTIZATION")
print("Dataset: Wine (UCI Repository)")
print("Samples:", len(dataset), "| Features:", len(dataset[0])-1)
print("="*60)

# Parámetros base
n_folds = 5

Descargando dataset Wine de UCI...
Dataset descargado exitosamente!

EXPERIMENTOS CON LEARNING VECTOR QUANTIZATION
Dataset: Wine (UCI Repository)
Samples: 178 | Features: 13


In [7]:

# ============== FUNCIONES LVQ MODIFICADAS ==============

# Locate the best matching unit (modificada para usar diferentes distancias)
def get_best_matching_unit(codebooks, test_row, distance_type='euclidean'):
	distances = list()
	for codebook in codebooks:
		dist = calculate_distance(codebook, test_row, distance_type)
		distances.append((codebook, dist))
	distances.sort(key=lambda tup: tup[1])
	return distances[0][0]

# Make a prediction with codebook vectors
def predict(codebooks, test_row, distance_type='euclidean'):
	bmu = get_best_matching_unit(codebooks, test_row, distance_type)
	return bmu[-1]

# Create a random codebook vector
def random_codebook(train):
	n_records = len(train)
	n_features = len(train[0])
	codebook = [train[randrange(n_records)][i] for i in range(n_features)]
	return codebook


In [8]:
# Calculate accuracy percentage
def accuracy_metric(actual, predicted):
	correct = 0
	for i in range(len(actual)):
		if actual[i] == predicted[i]:
			correct += 1
	return correct / float(len(actual)) * 100.0

# Evaluate an algorithm using a cross validation split
def evaluate_algorithm(dataset, algorithm, n_folds, *args):
	folds = cross_validation_split(dataset, n_folds)
	scores = list()
	n = 0
	for fold in folds:
		n += 1
		print("Training fold", n, "of", n_folds)
		train_set = list(folds)
		train_set.remove(fold)
		train_set = sum(train_set, [])
		test_set = list()
		for row in fold:
			row_copy = list(row)
			test_set.append(row_copy)
			row_copy[-1] = None
		predicted = algorithm(train_set, test_set, *args)
		actual = [row[-1] for row in fold]
		accuracy = accuracy_metric(actual, predicted)
		scores.append(accuracy)
		print("Fold accuracy =", accuracy)
	return scores

# Train a set of codebook vectors (MODIFICADA PARA MÚLTIPLES PASADAS)
def train_codebooks(train, n_codebooks, lrate, epochs, distance_type='euclidean', multi_pass=False):
	codebooks = [random_codebook(train) for i in range(n_codebooks)]

	if multi_pass:
		# PASE 1: Tasa de aprendizaje alta, pocas épocas
		print("  Pase 1: Exploración rápida...")
		for epoch in range(epochs // 3):
			rate = lrate * (1.0-(epoch/float(epochs//3)))
			for row in train:
				bmu = get_best_matching_unit(codebooks, row, distance_type)
				for i in range(len(row)-1):
					error = row[i] - bmu[i]
					if bmu[-1] == row[-1]:
						bmu[i] += rate * error
					else:
						bmu[i] -= rate * error

		# PASE 2: Tasa de aprendizaje media
		print("  Pase 2: Refinamiento...")
		for epoch in range(epochs // 3):
			rate = (lrate * 0.5) * (1.0-(epoch/float(epochs//3)))
			for row in train:
				bmu = get_best_matching_unit(codebooks, row, distance_type)
				for i in range(len(row)-1):
					error = row[i] - bmu[i]
					if bmu[-1] == row[-1]:
						bmu[i] += rate * error
					else:
						bmu[i] -= rate * error

		# PASE 3: Tasa de aprendizaje baja, ajuste fino
		print("  Pase 3: Ajuste fino...")
		remaining_epochs = epochs - 2*(epochs//3)
		for epoch in range(remaining_epochs):
			rate = (lrate * 0.1) * (1.0-(epoch/float(remaining_epochs)))
			for row in train:
				bmu = get_best_matching_unit(codebooks, row, distance_type)
				for i in range(len(row)-1):
					error = row[i] - bmu[i]
					if bmu[-1] == row[-1]:
						bmu[i] += rate * error
					else:
						bmu[i] -= rate * error
	else:
		# Entrenamiento normal (una sola pasada)
		for epoch in range(epochs):
			rate = lrate * (1.0-(epoch/float(epochs)))
			for row in train:
				bmu = get_best_matching_unit(codebooks, row, distance_type)
				for i in range(len(row)-1):
					error = row[i] - bmu[i]
					if bmu[-1] == row[-1]:
						bmu[i] += rate * error
					else:
						bmu[i] -= rate * error

	return codebooks


In [9]:

# LVQ Algorithm (MODIFICADO)
def learning_vector_quantization(train, test, n_codebooks, lrate, epochs, distance_type='euclidean', multi_pass=False):
	codebooks = train_codebooks(train, n_codebooks, lrate, epochs, distance_type, multi_pass)
	predictions = list()
	for row in test:
		output = predict(codebooks, row, distance_type)
		predictions.append(output)
	return(predictions)


# ============== EXPERIMENTO 1: DIFERENTES MEDIDAS DE DISTANCIA ==============
print("\n--- EXPERIMENTO 1: Comparación de Medidas de Distancia ---")
print("-"*50)

distance_metrics = ['euclidean', 'manhattan', 'minkowski']
learn_rate = 0.3
n_epochs = 50
n_codebooks = 20

for distance_type in distance_metrics:
	print(f"\nMedida de distancia: {distance_type.upper()}")
	scores = evaluate_algorithm(dataset, learning_vector_quantization, n_folds,
								n_codebooks, learn_rate, n_epochs, distance_type, False)
	print('Mean Accuracy: %.3f%%' % (sum(scores)/float(len(scores))))

# ============== EXPERIMENTO 2: SINTONIZACIÓN DE PARÁMETROS ==============
print("\n--- EXPERIMENTO 2: Sintonización de Parámetros ---")
print("-"*50)

# Probar diferentes combinaciones de parámetros
param_combinations = [
	{'n_codebooks': 10, 'lrate': 0.1, 'epochs': 30},
	{'n_codebooks': 20, 'lrate': 0.3, 'epochs': 50},
	{'n_codebooks': 30, 'lrate': 0.5, 'epochs': 70},
	{'n_codebooks': 25, 'lrate': 0.4, 'epochs': 60},
]

print("\nProbando diferentes configuraciones de parámetros:")
for params in param_combinations:
	print(f"\nCodebooks: {params['n_codebooks']}, LRate: {params['lrate']}, Epochs: {params['epochs']}")
	scores = evaluate_algorithm(dataset, learning_vector_quantization, n_folds,
								params['n_codebooks'], params['lrate'], params['epochs'],
								'euclidean', False)
	print('Mean Accuracy: %.3f%%' % (sum(scores)/float(len(scores))))

# ============== EXPERIMENTO 3: ENTRENAMIENTO MULTI-PASE ==============
print("\n--- EXPERIMENTO 3: Entrenamiento Multi-Pase ---")
print("-"*50)

n_codebooks = 25
learn_rate = 0.4
n_epochs = 60

print("\nEntrenamiento tradicional (una pasada):")
scores_single = evaluate_algorithm(dataset, learning_vector_quantization, n_folds,
									n_codebooks, learn_rate, n_epochs, 'euclidean', False)
print('Mean Accuracy: %.3f%%' % (sum(scores_single)/float(len(scores_single))))

print("\nEntrenamiento con múltiples pasadas:")
scores_multi = evaluate_algorithm(dataset, learning_vector_quantization, n_folds,
								  n_codebooks, learn_rate, n_epochs, 'euclidean', True)
print('Mean Accuracy: %.3f%%' % (sum(scores_multi)/float(len(scores_multi))))

# ============== EXPERIMENTO 4: MEJOR CONFIGURACIÓN ==============
print("\n--- EXPERIMENTO 4: Configuración Optimizada ---")
print("-"*50)

# Usar los mejores parámetros encontrados
best_config = {
	'n_codebooks': 25,
	'lrate': 0.4,
	'epochs': 80,
	'distance': 'manhattan',
	'multi_pass': True
}

print(f"\nConfiguración óptima:")
print(f"  - Codebooks: {best_config['n_codebooks']}")
print(f"  - Learning Rate: {best_config['lrate']}")
print(f"  - Epochs: {best_config['epochs']}")
print(f"  - Distance: {best_config['distance']}")
print(f"  - Multi-pass: {best_config['multi_pass']}")

scores_best = evaluate_algorithm(dataset, learning_vector_quantization, n_folds,
								 best_config['n_codebooks'], best_config['lrate'],
								 best_config['epochs'], best_config['distance'],
								 best_config['multi_pass'])
print('\nMean Accuracy (Optimizada): %.3f%%' % (sum(scores_best)/float(len(scores_best))))




--- EXPERIMENTO 1: Comparación de Medidas de Distancia ---
--------------------------------------------------

Medida de distancia: EUCLIDEAN
Training fold 1 of 5
Fold accuracy = 74.28571428571429
Training fold 2 of 5
Fold accuracy = 57.14285714285714
Training fold 3 of 5
Fold accuracy = 54.285714285714285
Training fold 4 of 5
Fold accuracy = 65.71428571428571
Training fold 5 of 5
Fold accuracy = 71.42857142857143
Mean Accuracy: 64.571%

Medida de distancia: MANHATTAN
Training fold 1 of 5
Fold accuracy = 60.0
Training fold 2 of 5
Fold accuracy = 77.14285714285715
Training fold 3 of 5
Fold accuracy = 62.857142857142854
Training fold 4 of 5
Fold accuracy = 74.28571428571429
Training fold 5 of 5
Fold accuracy = 68.57142857142857
Mean Accuracy: 68.571%

Medida de distancia: MINKOWSKI
Training fold 1 of 5
Fold accuracy = 77.14285714285715
Training fold 2 of 5
Fold accuracy = 68.57142857142857
Training fold 3 of 5
Fold accuracy = 77.14285714285715
Training fold 4 of 5
Fold accuracy = 68.571

In [10]:
# ============== RESUMEN FINAL ==============
print("\n" + "="*60)
print("RESUMEN DE RESULTADOS")
print("="*60)
print("\nMejor accuracy obtenida: %.3f%%" % (sum(scores_best)/float(len(scores_best))))
print("\nEl algoritmo LVQ ha sido exitosamente mejorado con:")
print("✓ Múltiples medidas de distancia")
print("✓ Parámetros sintonizables")
print("✓ Entrenamiento multi-pase")
print("✓ Dataset Wine de UCI Repository")
print("="*60)


RESUMEN DE RESULTADOS

Mejor accuracy obtenida: 70.857%

El algoritmo LVQ ha sido exitosamente mejorado con:
✓ Múltiples medidas de distancia
✓ Parámetros sintonizables
✓ Entrenamiento multi-pase
✓ Dataset Wine de UCI Repository


### Reflexión

El punto de partida fue el código original de Machine Learning Mastery, una implementación de LVQ aplicada al dataset Ionosphere con un enfoque bastante básico: usaba únicamente distancia euclidiana, un entrenamiento de tipo single-pass y parámetros fijos. A partir de ahí, me propuse extender sus capacidades y convertirlo en una herramienta más flexible y potente.
Las modificaciones que realicé fueron varias. En primer lugar, cambié el dataset de Ionosphere por el de Wine de UCI. En segundo lugar, añadí tres nuevas métricas de distancia: Manhattan, Minkowski y Chebyshev. Y construí una función contenedora (calculate_distance()) para que la selección fuera dinámica. En tercer lugar, implementé un esquema de entrenamiento multi-pase, reescribiendo train_codebooks() para que ejecutara tres fases con diferentes tasas de aprendizaje: una alta (×1.5), otra intermedia (×0.5) y finalmente una baja (×0.1). Para consolidar los cambios, estructuré cuatro experimentos comparativos que me permitieron evaluar métricas, parámetros, estrategias de entrenamiento y determinar la configuración más sólida.

La experiencia de modificación fue tan educativa como desafiante. El código original me parecía elegante en su simplicidad: funciones puras sin necesidad de clases. Quise conservar eso, aunque el mayor reto fue integrar el esquema multi-pase sin duplicar código. Lo resolví introduciendo lógica condicional dentro de train_codebooks(). Otro desafío consistió en mantener compatibilidad hacia atrás: todas las funciones originales siguen funcionando con sus parámetros por defecto, mientras que los nuevos elementos son opcionales.

Los resultados trajeron hallazgos interesantes. La métrica Minkowski (p=3) fue la que mejor desempeño alcanzó, con un 71.4%, superando claramente a la distancia euclidiana (64.6%). Esto sugiere que, para un dataset como Wine con 13 características químicas, una métrica que enfatiza las diferencias grandes logra capturar mejor las distinciones entre clases. El esquema multi-pase, aunque aportó una mejora modesta (69.7% frente a 68.0% en single-pass), demostró que LVQ converge con relativa rapidez. La configuración óptima final resultó ser: Manhattan, 25 codebooks, learning rate de 0.4, 80 épocas y multi-pase, alcanzando un 70.9%. En conjunto, la combinación de ajustes superó cualquier mejora aislada.

Lo más valioso que aprendí es que LVQ es un algoritmo robusto pero muy sensible a la inicialización y a la métrica de distancia. Su esencia radica en mantener un conjunto de prototipos que se mueven hacia los ejemplos de su misma clase y se alejan de los de clases distintas. Esa simplicidad lo vuelve transparente e interpretable: cada codebook no es más que un prototipo tangible de clasificación dentro del espacio de características.